# 意味で探す蔵書検索

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/funakoshi-takehiro/library-hiroba/blob/main/notebooks/book_search.ipynb)

「怖い本を教えて」と書くだけで、**言葉が一致していなくても意味の近い本**が見つかります。
`ai.search()` が、文を「意味を表す数のならび（ベクトル）」に変えて、近いものを探します。

ふつうの検索は文字が一致するかを見るので、「怖い」と書いてある本しか見つかりません。
こちらは**意味**を見るので、説明文に「怪奇現象」としか書いていない本も見つかります。

**Google Colab でも PyHiroba でも同じように動きます。**

In [ ]:
%pip install -q -U "library-hiroba[ai]"
# PyHiroba では、このセルの実行は不要です

# このノートブックが前提にしている版。ここより古いと ai.search が入っていません
NEEDS = "0.6.0"

import sys
from importlib.metadata import PackageNotFoundError, version


def numbers(text):
    return tuple(int(part) for part in str(text).split(".")[:3])


try:
    installed = version("library-hiroba")
except PackageNotFoundError:
    installed = None  # PyHiroba は同梱なので、配布情報を持ちません
loaded = getattr(sys.modules.get("library_hiroba"), "__version__", None)

if installed and loaded and loaded != installed:
    print(f"{loaded} が読み込まれたままなので、セッションを再起動します。")
    print(f"再起動したら、このセルから順に実行し直してください（{installed} になります）。")
    try:
        import IPython

        IPython.Application.instance().kernel.do_shutdown(True)
    except Exception:
        print("自動で再起動できませんでした。")
        print("メニューの「ランタイム」→「セッションを再起動する」を実行してください。")
elif installed and numbers(installed) < numbers(NEEDS):
    print(f"このノートブックは {NEEDS} 以降を前提にしていますが、{installed} が入っています。")
    print("まだ公開されていない版を試すときは、次を実行してから再起動してください:")
    print('  %pip install -q -U "git+https://github.com/funakoshi-takehiro/library-hiroba@main"')
else:
    print("library-hiroba", installed or loaded)

## 本を用意する

10冊ぶんの題名と説明文です。説明文だけを検索の対象にします。

In [ ]:
from library_hiroba import ai, ui

books = [
    {"title": "真夜中の校舎",     "desc": "深夜の学校に閉じ込められた生徒。次々と起きる怪異の記録。"},
    {"title": "はじめての料理",   "desc": "包丁の持ち方から、やさしく学べる入門書。"},
    {"title": "宇宙のはなし",     "desc": "星はどうやって生まれるのか。銀河と惑星のしくみ。"},
    {"title": "走る科学",         "desc": "速く走るための体の使い方を、運動生理学から説明する。"},
    {"title": "古い洋館の秘密",   "desc": "誰もいないはずの部屋から足音がする。屋敷にまつわる言い伝え。"},
    {"title": "パンを焼く",       "desc": "家庭のオーブンでふっくら焼くための、粉と発酵の話。"},
    {"title": "プログラミング入門", "desc": "順次・分岐・繰り返し。はじめて書く人のための考え方。"},
    {"title": "日本の四季",       "desc": "俳句とともにたどる、春夏秋冬の移り変わり。"},
    {"title": "深海の生きもの",   "desc": "光の届かない海にすむ、奇妙なすがたの生物たち。"},
    {"title": "友だちのつくり方", "desc": "話しかけるのが苦手な人へ。少しずつ人と関わるための本。"},
]

ui.table([{"題名": b["title"], "説明": b["desc"]} for b in books], caption="蔵書")

## 意味で探す

`ai.search()` に「探したい言葉」と「探される文のリスト」を渡します。
近い順に `index`（何番目か）・`score`（近さ）・`text` が返ります。

初回はモデルの取得（約118MB）が入ります。PyHiroba では確認の画面が出ます。

In [ ]:
hits = await ai.search("怖い本を教えて", [b["desc"] for b in books], top_k=3)

ui.table(
    [{"順位": i + 1, "題名": books[h["index"]]["title"], "近さ": round(h["score"], 3)}
     for i, h in enumerate(hits)],
    caption="「怖い本を教えて」に近い本",
)

### いろいろな聞き方を試す

説明文に無い言葉で聞いても見つかることを確かめてみてください。

In [ ]:
for word in ["料理を作りたい", "運動が得意になりたい", "さびしい気持ち"]:
    found = await ai.search(word, [b["desc"] for b in books], top_k=1)
    print(f'{word:<16} → {books[found[0]["index"]]["title"]}（{found[0]["score"]:.3f}）')

## しくみを見る

`ai.search()` の中身は、`ai.embed()` と掛け算だけです。

`ai.embed()` は文をベクトルにします。長さが 1 にそろえてある（正規化済み）ので、
**2つのベクトルを掛けて足すと、そのまま「近さ」になります**（コサイン類似度）。

In [ ]:
vectors = await ai.embed([b["desc"] for b in books])
question = await ai.embed("怖い本を教えて")

print("ベクトルの長さ:", len(question), "個の数")
print("先頭の5個:", [round(v, 3) for v in question[:5]])

# 掛けて足す（内積）だけで近さが出る
scores = [(sum(q * v for q, v in zip(question, vec)), b["title"])
          for vec, b in zip(vectors, books)]
for score, title in sorted(scores, reverse=True)[:3]:
    print(f"  {score:.3f}  {title}")

---

## 気をつけること

- **索引と検索は、同じ環境で作ってください。** ブラウザ（PyHiroba）は int8、
  Colab は fp32 で計算するため、同じ文でもベクトルがわずかに違います。
  同じ環境の中で比べるぶんには問題ありませんが、Colab で作ったベクトルに
  PyHiroba の検索を混ぜると、順位が変わることがあります。
- **説明文は短めに。** 1文は 512 トークンで切り詰められます。本の紹介文（数文）なら
  余裕ですが、本文をまるごと入れると先頭だけで判断されます。
- 件数が多くても、そのまま渡して構いません（内部で分けて処理します）。

確認ポイント:

- 「怖い本を教えて」で `真夜中の校舎` と `古い洋館の秘密` が上位に来る
- 説明文に「料理」と書いていない `パンを焼く` も、「料理を作りたい」で見つかる
- `ai.embed()` が 384 個の数を返し、内積が `ai.search()` の `score` と一致する

モデルのライセンスは配布元をご確認ください
（`paraphrase-multilingual-MiniLM-L12-v2` は Apache-2.0）。